<a href="https://colab.research.google.com/github/Madathanapalleleena/DL_exploration/blob/main/Pretrained_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
zip_path = "/content/drive/MyDrive/dataset/archive.zip"

In [3]:
import zipfile

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/archive")

In [4]:
import os

data_path = "/content/archive"
print(os.listdir(data_path))

['brain_tumor_dataset', 'yes', 'no']


In [6]:
import os

base_path = "/content/archive"

for item in os.listdir(base_path):
    print(item, "->", os.listdir(os.path.join(base_path, item))[:5])

brain_tumor_dataset -> ['yes', 'no']
yes -> ['Y71.JPG', 'Y259.JPG', 'Y167.JPG', 'Y250.jpg', 'Y67.JPG']
no -> ['no 4.jpg', '22 no.jpg', 'no 1.jpg', '31 no.jpg', 'no 3.jpg']


In [7]:
base_path = "/content/archive/brain_tumor_dataset"

In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (128, 128)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [10]:
print(train_data.class_indices)

{'no': 0, 'yes': 1}


In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    base_path,
    target_size=img_size,
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    base_path,
    target_size=img_size,
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 203 images belonging to 2 classes.
Found 50 images belonging to 2 classes.


In [19]:
import tensorflow as tf
import keras
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, Flatten, Dense, Dropout

In [20]:
data_path = "/content/archive/brain_tumor_dataset"

categories = ["no", "yes"]
img_size = 128

data = []
labels = []

for category in categories:
    path = os.path.join(data_path, category)
    label = categories.index(category)

    for img in os.listdir(path):
        try:
            img_array = cv2.imread(os.path.join(path, img))
            img_array = cv2.resize(img_array, (img_size, img_size))
            data.append(img_array)
            labels.append(label)
        except:
            pass

data = np.array(data) / 255.0
labels = np.array(labels)

In [22]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2)

# further split validation
x_train_main, x_val, y_train_main, y_val = train_test_split(x_train, y_train, test_size=0.2)

x_train_main = x_train_main.reshape(-1, 128, 128, 3)
x_val = x_val.reshape(-1, 128, 128, 3)
x_test = x_test.reshape(-1, 128, 128, 3)

In [23]:
LeNet = Sequential()

LeNet.add(Conv2D(6, (5,5), activation='tanh', input_shape=(128,128,3)))
LeNet.add(AveragePooling2D((2,2)))

LeNet.add(Conv2D(16, (5,5), activation='tanh'))
LeNet.add(AveragePooling2D((2,2)))

LeNet.add(Flatten())

LeNet.add(Dense(120, activation='tanh'))
LeNet.add(Dense(84, activation='tanh'))
LeNet.add(Dense(1, activation='sigmoid'))

LeNet.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

LeNet.fit(x_train_main, y_train_main, epochs=10, batch_size=32, validation_data=(x_val, y_val))

print("LeNet Accuracy:", LeNet.evaluate(x_test, y_test)[1])

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 359ms/step - accuracy: 0.5528 - loss: 1.1556 - val_accuracy: 0.6585 - val_loss: 0.5727
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 314ms/step - accuracy: 0.6770 - loss: 0.5239 - val_accuracy: 0.7317 - val_loss: 0.5260
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 316ms/step - accuracy: 0.6957 - loss: 0.5781 - val_accuracy: 0.7317 - val_loss: 0.5478
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 346ms/step - accuracy: 0.8199 - loss: 0.5120 - val_accuracy: 0.7317 - val_loss: 0.5862
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 376ms/step - accuracy: 0.7950 - loss: 0.5239 - val_accuracy: 0.7073 - val_loss: 0.5329
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 554ms/step - accuracy: 0.7826 - loss: 0.4373 - val_accuracy: 0.7073 - val_loss: 0.5739
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 323ms/step - accuracy: 0.8323 - loss: 0.4005 - val_accuracy: 0.7073 - val_loss: 0.5146
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 323ms/step - accuracy: 0.8634 - loss: 0.3360 - val_accuracy: 0.7561 - val_loss:

In [24]:
AlexNet = Sequential()

AlexNet.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
AlexNet.add(MaxPooling2D((2,2)))

AlexNet.add(Conv2D(64, (3,3), activation='relu'))
AlexNet.add(MaxPooling2D((2,2)))

AlexNet.add(Conv2D(128, (3,3), activation='relu'))
AlexNet.add(MaxPooling2D((2,2)))

AlexNet.add(Flatten())

AlexNet.add(Dense(256, activation='relu'))
AlexNet.add(Dropout(0.5))
AlexNet.add(Dense(1, activation='sigmoid'))

AlexNet.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

AlexNet.fit(x_train_main, y_train_main, epochs=10, batch_size=32, validation_data=(x_val, y_val))

print("AlexNet Accuracy:", AlexNet.evaluate(x_test, y_test)[1])

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 852ms/step - accuracy: 0.6149 - loss: 1.0719 - val_accuracy: 0.7317 - val_loss: 0.6067
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 943ms/step - accuracy: 0.7578 - loss: 0.6345 - val_accuracy: 0.7561 - val_loss: 0.5537
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 846ms/step - accuracy: 0.7826 - loss: 0.5019 - val_accuracy: 0.7317 - val_loss: 0.5650
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.8012 - loss: 0.4660 - val_accuracy: 0.7073 - val_loss: 0.5885
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 869ms/step - accuracy: 0.7950 - loss: 0.4684 - val_accuracy: 0.7073 - val_loss: 0.6052
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 895ms/step - accuracy: 0.8199 - loss: 0.4506 - val_accuracy: 0.7317 - val_loss: 0.5495
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.8075 - loss: 0.4484 - val_accuracy: 0.7317 - val_loss: 0.6049
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 742ms/step - accuracy: 0.8447 - loss: 0.4180 - val_accuracy: 0.7073 - val_loss: 0.57

In [26]:
base = tf.keras.applications.VGG16(include_top=False, input_shape=(128,128,3), weights='imagenet')
base.trainable = False

VGG = Sequential([
    base,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

VGG.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

VGG.fit(x_train_main, y_train_main, epochs=5, batch_size=32, validation_data=(x_val, y_val))

print("VGG Accuracy:", VGG.evaluate(x_test, y_test)[1])

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 37s 6s/step - accuracy: 0.6584 - loss: 1.1396 - val_accuracy: 0.3902 - val_loss: 1.7035
Epoch 2/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 43s 8s/step - accuracy: 0.6273 - loss: 1.0842 - val_accuracy: 0.6098 - val_loss: 1.0800
Epoch 3/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 77s 6s/step - accuracy: 0.7826 - loss: 0.5399 - val_accuracy: 0.5610 - val_loss: 0.7707
Epoch 4/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 37s 6s/step - accuracy: 0.7391 - loss: 0.6180 - val_accuracy: 0.7805 - val_loss: 0.5216
Epoch 5/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 41s 6s/step - accuracy: 0.8571 - loss: 0.3292 - val_accuracy: 0.8780 - val_loss: 0.3531
2/2 ━━━━━━━━━━━━━━━━━━━━ 8s 3s/step - accuracy: 0.7647 - loss: 0.4828
VGG Accuracy: 0.7647058963775635


In [25]:
base = tf.keras.applications.ResNet50(include_top=False, input_shape=(128,128,3), weights='imagenet')
base.trainable = False

ResNet = Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

ResNet.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

ResNet.fit(x_train_main, y_train_main, epochs=5, batch_size=32, validation_data=(x_val, y_val))

print("ResNet Accuracy:", ResNet.evaluate(x_test, y_test)[1])

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 28s 3s/step - accuracy: 0.5963 - loss: 0.7274 - val_accuracy: 0.6098 - val_loss: 0.6929
Epoch 2/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.6460 - loss: 0.6844 - val_accuracy: 0.6098 - val_loss: 0.6789
Epoch 3/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.6522 - loss: 0.6477 - val_accuracy: 0.6098 - val_loss: 0.6587
Epoch 4/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.5714 - loss: 0.6890 - val_accuracy: 0.6585 - val_loss: 0.6766
Epoch 5/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.5590 - loss: 0.6865 - val_accuracy: 0.7073 - val_loss: 0.6677
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 828ms/step - accuracy: 0.5882 - loss: 0.6750
ResNet Accuracy: 0.5882353186607361


In [27]:
base = tf.keras.applications.InceptionV3(include_top=False, input_shape=(128,128,3), weights='imagenet')
base.trainable = False

Inception = Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

Inception.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

Inception.fit(x_train_main, y_train_main, epochs=5, batch_size=32, validation_data=(x_val, y_val))

print("Inception Accuracy:", Inception.evaluate(x_test, y_test)[1])

Epoch 1/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.5652 - loss: 3.4615 - val_accuracy: 0.6585 - val_loss: 0.6003
Epoch 2/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 889ms/step - accuracy: 0.5528 - loss: 0.9794 - val_accuracy: 0.6829 - val_loss: 0.5408
Epoch 3/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.6584 - loss: 0.6704 - val_accuracy: 0.6098 - val_loss: 0.5757
Epoch 4/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 885ms/step - accuracy: 0.6646 - loss: 0.6066 - val_accuracy: 0.8293 - val_loss: 0.4802
Epoch 5/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.7640 - loss: 0.4901 - val_accuracy: 0.8537 - val_loss: 0.4498
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 501ms/step - accuracy: 0.8039 - loss: 0.5075
Inception Accuracy: 0.8039215803146362


LeNet: Lowest accuracy (simple model)

AlexNet: Better but may overfit

VGG16: High accuracy due to transfer learning

ResNet: Most stable performance

Inception: Good feature extraction

Best Model: ResNet / VGG16

Reason: Pre-trained on large dataset → better feature learning